<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/05_surrogate_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 5: Surrogate Modelling with OpenImpala-Generated Data

Training a machine learning model to predict material transport properties requires a labelled dataset: many 3D structures paired with their ground-truth physics. OpenImpala can generate these labels efficiently, making it practical to build surrogate models that approximate the full PDE solve at a fraction of the cost.

**Connection to Tutorial 3 (REV):** In the [REV tutorial](03_rev_and_uncertainty.ipynb) we showed that small sub-volumes exhibit large variance in tortuosity, and that this variance is physically meaningful — it reflects real structure–property correlations. Here we take advantage of that: each small crop with its unique porosity and tortuosity becomes a training sample.

**What you will learn:**
1. Generate randomised microstructures using PoreSpy.
2. Compute morphological features (porosity, specific surface area).
3. Use OpenImpala to obtain ground-truth tortuosity for each sample.
4. Train a Random Forest regressor and evaluate its accuracy.

In [ ]:
# Install OpenImpala and ML libraries
!pip install -q openimpala porespy scikit-learn matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import porespy as ps
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded.")

N_SAMPLES = 50   # Number of random microstructures to generate
SIZE = 32         # Voxel side length of each sample

In [ ]:
print("Step 1: Generating labelled data with OpenImpala...")
X_features = []  # [Porosity, Specific Surface Area]
y_target = []    # [Tortuosity]

t_start = time.time()

# Use a single Session for the whole loop to avoid repeated MPI init/finalize.
with oi.Session():
    for i in range(N_SAMPLES):
        # 1. Random blobby structure with varying porosity and blob size
        target_porosity = np.random.uniform(0.3, 0.7)
        blobiness = np.random.uniform(1.0, 3.0)
        micro = ps.generators.blobs(shape=[SIZE, SIZE, SIZE], porosity=target_porosity, blobiness=blobiness)
        micro = micro.astype(np.int32)

        # 2. Morphological features
        vf = oi.volume_fraction(micro, phase=1).fraction
        surface_area = ps.metrics.specific_surface_area(micro)

        # 3. Ground-truth physics from OpenImpala
        perc = oi.percolation_check(micro, phase=1, direction="z")
        if perc.percolates:
            tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity
            X_features.append([vf, surface_area])
            y_target.append(tau)

X_features = np.array(X_features)
y_target = np.array(y_target)

t_data = time.time() - t_start
print(f"Generated {len(y_target)} valid samples in {t_data:.2f}s.")

In [ ]:
print("\nStep 2: Training a Random Forest surrogate...")
X_train, X_test, y_train, y_test = train_test_split(X_features, y_target, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

t0 = time.time()
y_pred = model.predict(X_test)
t_predict = time.time() - t0

r2 = r2_score(y_test, y_pred)
print(f"R² score: {r2:.3f}")
print(f"Prediction time for {len(X_test)} samples: {t_predict:.4f}s")

# True vs. predicted
fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
ax.scatter(y_test, y_pred, color='#1f77b4', s=50, alpha=0.8, edgecolor='black')
ax.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', lw=2, label="Perfect prediction")
ax.set_xlabel("True Tortuosity (OpenImpala)")
ax.set_ylabel("Predicted Tortuosity (Random Forest)")
ax.set_title("Surrogate Model: True vs. Predicted")
ax.legend()
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Next Steps

This tutorial used OpenImpala as a data-generation engine to label microstructures with ground-truth transport properties, then trained a simple surrogate model. With more samples and richer features (e.g. two-point correlation functions, chord-length distributions), the same approach can yield accurate surrogates for rapid screening of candidate microstructures.

**Related tutorials:**
- [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb) — Put OpenImpala inside a design loop to optimise microstructure geometry.
- [Tutorial 7: Scaling to HPC](07_hpc_scaling.ipynb) — Generate larger datasets on a cluster with MPI.